In [ ]:
import pandas as pd
import joblib
import os
import yaml
import mlflow
from mlflow.models import infer_signature
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error , mean_absolute_error , r2_score

In [3]:
data = yaml.safe_load(open("../params.yaml"))["model"]

df = pd.read_csv(data["input"])

df.head()

,ID,Square_Feet,Num_Bedrooms,Num_Bathrooms,Num_Floors,Year_Built,Has_Garden,Has_Pool,Garage_Size,Location_Score,Distance_to_Center,Price
0,1,143.64,1,3,3,1967,1,1,48,8.30,5.94,602134.82
1,2,287.68,1,2,1,1949,0,1,37,6.06,10.83,591425.14
2,3,233.00,1,3,2,1923,1,0,14,2.91,6.90,464478.70
3,4,199.66,5,2,2,1918,0,0,17,2.07,8.28,583105.66
4,5,89.00,4,3,3,1999,1,0,34,1.52,14.65,619879.14


## Train and Test

In [4]:
X = df.drop(["Price" , "ID"] , axis = 1)
Y = df["Price"]

In [5]:
xtrain , xtest , ytrain , ytest = train_test_split(X , Y , test_size = 0.3 , random_state = 50)

## Model training and evaluation

In [6]:
#parameter
params_grid1 = {
    "n_estimators" : data["parameter_model"]["RandomForest"]["n_estimators"],
    "max_depth" : data["parameter_model"]["RandomForest"]["max_depth"],
    "min_samples_split" : data["parameter_model"]["RandomForest"]["min_samples_split"],
    "min_samples_leaf" : data["parameter_model"]["RandomForest"]["min_samples_leaf"],
}

params_grid2 = {
    "max_depth" : data["parameter_model"]["DecisionTree"]["max_depth"],
    "min_samples_split" : data["parameter_model"]["DecisionTree"]["min_samples_split"],
    "min_samples_leaf" : data["parameter_model"]["DecisionTree"]["min_samples_leaf"],
}

In [7]:
def grid_search(model ,x_train , y_train , params_grid = None):
    if params_grid is None:
        params_grid = {}
    grid = GridSearchCV(estimator=model , param_grid=params_grid ,
                         cv=3 , n_jobs=-1 , verbose=2)
    grid.fit(x_train , y_train)
    return grid


best_linear_regression = grid_search(LinearRegression(),xtrain , ytrain)
best_random_forest = grid_search(RandomForestRegressor(random_state=50),xtrain,ytrain,params_grid1)
best_decision_tree = grid_search(DecisionTreeRegressor(random_state=45),xtrain,ytrain,params_grid2)

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 360 candidates, totalling 1080 fits
Fitting 3 folds for each of 180 candidates, totalling 540 fits


In [8]:
best_estimator_regression = best_linear_regression.best_estimator_
best_estimator_forest = best_random_forest.best_estimator_
best_estimator_tree = best_decision_tree.best_estimator_

best_params_forest = best_random_forest.best_params_
best_params_tree = best_decision_tree.best_params_

In [9]:
def evaluate(error_score , model , y_test):
    ypred = model.predict(xtest)
    return error_score(y_test , ypred)

In [10]:
squared_regression = evaluate(mean_squared_error , best_estimator_regression, ytest)
absolute_regression = evaluate(mean_absolute_error , best_estimator_regression , ytest)
r2_regression = evaluate(r2_score , best_estimator_regression , ytest)

squared_forest = evaluate(mean_squared_error , best_estimator_forest, ytest)
absolute_forest = evaluate(mean_absolute_error , best_estimator_forest , ytest)
r2_forest = evaluate(r2_score , best_estimator_forest , ytest)


squared_tree = evaluate(mean_squared_error , best_estimator_tree, ytest)
absolute_tree = evaluate(mean_absolute_error , best_estimator_tree , ytest)
r2_tree = evaluate(r2_score , best_estimator_tree , ytest)

In [ ]:
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

signature = infer_signature(xtrain , ytrain)

with mlflow.start_run():
    # mlflow for LinearRegression
    mlflow.log_metric("LinearRegression_squared" , squared_regression)
    mlflow.log_metric("LinearRegression_absolute" , absolute_regression)
    mlflow.log_metric("LinearRegression_r2" , r2_regression)

    model = urlparse(mlflow.get_tracking_uri()).scheme

    if model != "file":
        mlflow.sklearn.log_model(best_estimator_regression,
                                 "model_linear_regression",
                                 registered_model_name="LinearRegression")
    else:
        mlflow.sklearn.log_model(best_estimator_regression,
                                 "model_linear_regression",
                                 signature=signature)


with mlflow.start_run():
    # mlflow for DecisionTreeRegressor
    mlflow.log_metric("DecisionTreeRegressor_squared" , squared_tree)
    mlflow.log_metric("DecisionTreeRegressor_absolute" , absolute_tree)
    mlflow.log_metric("DecisionTreeRegressor_r2" , r2_tree)

    mlflow.log_param("DecisionTreeRegressor_param" , best_params_tree)

    uri = urlparse(mlflow.get_tracking_uri()).scheme

    if uri != "file":
        mlflow.sklearn.log_model(best_estimator_tree,
                                 "model_DecisionTree_Regressor",
                                 registered_model_name="DecisionTreeRegressor",
                                 skops_trusted_types=["sklearn.tree._tree.Tree"])
    else:
        mlflow.sklearn.log_model(best_estimator_forest,
                                 "model_DecisionTree_Regressor",
                                 signature=signature)


with mlflow.start_run():
    # mlflow for RandomForestRegressor
    mlflow.log_metric("RandomForestRegressor_squared" , squared_forest)
    mlflow.log_metric("RandomForestRegressor_absolute" , absolute_forest)
    mlflow.log_metric("RandomForestRegressor_r2" , r2_forest)

    mlflow.log_param("RandomForestRegressor_param" , best_params_forest)

    uri = urlparse(mlflow.get_tracking_uri()).scheme

    if uri != "file":
        mlflow.sklearn.log_model(best_estimator_forest,
                                 "model_RandomForest_Regressor",
                                 registered_model_name="RandomForestRegressor",
                                 skops_trusted_types=["sklearn.tree._tree.Tree"])
    else:
        mlflow.sklearn.log_model(best_estimator_forest,
                                 "model_RandomForest_Regressor",
                                 signature=signature)

2026/10/26 03:29:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'LinearRegression' already exists. Creating a new version of this model...
2026/10/26 03:29:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LinearRegression, version 3
Created version '3' of model 'LinearRegression'.


🏃 View run bald-carp-0 at: http://127.0.0.1:5000/#/experiments/0/runs/a5f114789a5a4802969f3e205ded7639
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


2026/10/26 03:29:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'DecisionTreeRegressor'.
2026/10/26 03:30:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DecisionTreeRegressor, version 1
Created version '1' of model 'DecisionTreeRegressor'.


🏃 View run adventurous-worm-943 at: http://127.0.0.1:5000/#/experiments/0/runs/0f940c2267a24ec3bac61fbf1bf2cf80
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


2026/10/26 03:30:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'RandomForestRegressor'.
2026/10/26 03:30:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForestRegressor, version 1
Created version '1' of model 'RandomForestRegressor'.


🏃 View run youthful-kit-525 at: http://127.0.0.1:5000/#/experiments/0/runs/392fcbe524394a08a5730ebb02907a7a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


In [31]:
with open(data["output"],"wb") as r:
    joblib.dump(best_estimator_regression , r)